In [ ]:
import torch
from deep_speech_2 import DeepSpeech2
from deep_speech_2_bidirectional import DeepSpeech2Bidirectional
from deep_speech_2_lstm import DeepSpeech2LSTM
from conformer import Conformer
import utils

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

# select model
model = Conformer(conv_in_channels=1)
MODEL_PATH = 'models/seed_runs/conformer_18M/seed_2603/model_best.pth'
utils.load_model(model=model, device=device, filepath=MODEL_PATH)

In [ ]:
import os
from torchmetrics.text import WordErrorRate
from config import LM_ARPA_PATH, LEXICON_PATH, LM_WEIGHT, WORD_SCORE
from decoder import Decoder

decoder_no_lm = None
decoder_lm = None

if model is None:
    print("No active model selected. Skipping decoder setup.")
elif not os.path.exists(LM_ARPA_PATH):
    print(f"LM file not found: {LM_ARPA_PATH}")
elif not os.path.exists(LEXICON_PATH):
    print(f"Lexicon file not found: {LEXICON_PATH}")
else:
    decoder_no_lm = Decoder(
        tokenizer=model.tokenizer,
        blank_token=model.tokenizer.pad_token,
    )
    decoder_lm = Decoder(
        tokenizer=model.tokenizer,
        blank_token=model.tokenizer.pad_token,
        lexicon=LEXICON_PATH,
        lm=LM_ARPA_PATH,
        lm_weight=LM_WEIGHT,
        word_score=WORD_SCORE,
    )
    print("Decoders ready: baseline and LibriSpeech LM")

In [ ]:
def sweep_lm_params(
    data_loader,
    model,
    lexicon_path=LEXICON_PATH,
    lm_path=LM_ARPA_PATH,
    lm_weights=None,
    word_scores=None,
):
    if lm_weights is None:
        lm_weights  = [0.5, 1.0, 1.5]

    if word_scores is None:
        word_scores = [-1.0, 0.5]

    model.eval()
    cached_outputs = []
    with torch.no_grad():
        for batch in data_loader:
            specs = batch['padded_spectrograms'].to(device)
            seq_lens = batch['input_lengths']
            log_probs, out_lens = model(specs, seq_lens)
            cached_outputs.append((log_probs.cpu(), out_lens.cpu()))

    best_wer = float('inf')
    best_params = {'lm_weight': LM_WEIGHT, 'word_score': WORD_SCORE}
    results = []

    for lw in lm_weights:
        for ws in word_scores:
            sweep_decoder = Decoder(
                tokenizer=model.tokenizer,
                blank_token=model.tokenizer.pad_token,
                lexicon=lexicon_path,
                lm=lm_path,
                lm_weight=lw,
                word_score=ws,
            )
            wer_metric = WordErrorRate()
            for batch, (log_probs, out_lens) in zip(data_loader, cached_outputs):
                hyps = sweep_decoder.decode_beam(
                    specs=None,
                    spec_lengths=None,
                    model=model,
                    log_probs=log_probs,
                    out_lens=out_lens,
                )
                refs = [
                    model.tokenizer.decode(
                        batch['packed_transcripts'][
                            sum(batch['target_lengths'][:i]):
                            sum(batch['target_lengths'][:i + 1])
                        ].tolist()
                    )
                    for i in range(len(hyps))
                ]
                wer_metric.update(hyps, refs)

            wer = wer_metric.compute().item()
            results.append((lw, ws, wer))
            if wer < best_wer:
                best_wer = wer
                best_params = {'lm_weight': lw, 'word_score': ws}
            print(f"lm_weight={lw:.1f} word_score={ws:+.1f} val_WER={wer:.4f}")

    print(f"Best val WER {best_wer:.4f} at {best_params}")
    return best_wer, best_params, results

In [ ]:
# load dataset
import torchaudio
from config import Dataset
from ljspeech import load_ljspeech

DATASET = Dataset.LJSPEECH
CACHE_DIR = f"./data/cache/{DATASET.name.lower()}"

if (DATASET == Dataset.LJSPEECH or
    DATASET == Dataset.MIXED):
    lj_dataset = load_ljspeech("../data/LJSpeech-1.1/LJSpeech-1.1", cache_dir=CACHE_DIR)
    num_lj_samples = len(lj_dataset)

    _display_path = lj_dataset.file_paths[0]
    _display_audio, _display_sr = torchaudio.load(_display_path)
    _display_mel, _display_tokens = lj_dataset[0]

    # Split LJSpeech 80% train / 10% val /10% test
    # Test set = LibriSpeech test-clean only (keeps evaluation domain consistent)
    lj_train_size = int(num_lj_samples * 0.8)
    lj_val_size   = int(num_lj_samples * 0.1)
    lj_test_size  = num_lj_samples - lj_train_size - lj_val_size
    lj_train, lj_val, lj_test = torch.utils.data.random_split(
        lj_dataset, [lj_train_size, lj_val_size, lj_test_size],
        generator=torch.Generator().manual_seed(1508)
    )
    print(f"LJSpeech  — train: {len(lj_train)}, val: {len(lj_val)}, test: {len(lj_test)}")
    train_set = lj_train; val_set = lj_val; test_set = lj_test;

In [ ]:
BATCH_SIZE = 32
NUM_WORKERS = 0
PREFETCH_FACTOR = 2
PIN_MEMORY = True

_persistent = NUM_WORKERS > 0
_prefetch   = PREFETCH_FACTOR if NUM_WORKERS > 0 else None

train_loader = torch.utils.data.DataLoader(
    dataset=train_set,
    batch_size=BATCH_SIZE,
    collate_fn=utils.collate_fn_train,
    shuffle=True,
    num_workers=NUM_WORKERS,
    persistent_workers=_persistent,
    prefetch_factor=_prefetch,
    pin_memory=PIN_MEMORY,
 )
val_loader = torch.utils.data.DataLoader(
    dataset=val_set,
    batch_size=BATCH_SIZE,
    collate_fn=utils.collate_fn_eval,
    shuffle=False,
    num_workers=NUM_WORKERS,
    persistent_workers=_persistent,
    prefetch_factor=_prefetch,
    pin_memory=PIN_MEMORY,
 )
test_loader = torch.utils.data.DataLoader(
    dataset=test_set,
    batch_size=BATCH_SIZE,
    collate_fn=utils.collate_fn_eval,
    shuffle=False,
    num_workers=NUM_WORKERS,
    persistent_workers=_persistent,
    prefetch_factor=_prefetch,
    pin_memory=PIN_MEMORY,
 )

section_specs = next(iter(train_loader))['padded_spectrograms']
section_in_channels = section_specs.shape[1]
section_in_feat_dim = section_specs.shape[2]
print(f"section_in_channels={section_in_channels}, section_in_feat_dim={section_in_feat_dim}")
print(f"train/val/test batches: {len(train_loader)}/{len(val_loader)}/{len(test_loader)}")

In [ ]:
RUN_LM_SWEEP = True

best_lm_params = {'lm_weight': LM_WEIGHT, 'word_score': WORD_SCORE}

if RUN_LM_SWEEP and model is not None and decoder_lm is not None:
    _, best_lm_params, _ = sweep_lm_params(
        data_loader=val_loader,
        model=model,
        lexicon_path=LEXICON_PATH,
        lm_path=LM_ARPA_PATH,
    )
    print(f"Selected LM params: {best_lm_params}")
    decoder_lm = Decoder(
        tokenizer=model.tokenizer,
        blank_token=model.tokenizer.pad_token,
        lexicon=LEXICON_PATH,
        lm=LM_ARPA_PATH,
        lm_weight=best_lm_params['lm_weight'],
        word_score=best_lm_params['word_score'],
    )
else:
    print("Skipping LM sweep.")